# Initial EDA
Objective: better understand data and identify drawbacks of standard Euclidean distance metric

## Import

In [2]:
# Making sure all files that need to read each other are able to
import sys
from pathlib import Path
ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from paths import DUCKDB_PATH

# Import libraries
import duckdb
import pandas as pd
import numpy as np
import plotly.express as px

In [3]:
# Gate open - read_only to prevent accidental locking/corrupting the db while exploring
con = duckdb.connect(str(DUCKDB_PATH), read_only=True)

# Data Structure Check

In [5]:
# should be skills, pairs, real_transitions, pairs_staged
con.execute("SHOW TABLES").df() 

,name
0,pairs
1,pairs_staged
2,real_transitions
3,skills


In [8]:
# Grab what we want
pairs_staged_df = con.execute("SELECT * FROM pairs_staged").df()  # main table
skills_df = con.execute("SELECT * FROM skills").df()        # 35 skill vectors

In [ ]:
# plotly check
px.histogram(pairs_staged_df, x="wage_to").show()

# EDA

## Initial Checks: shape, dtypes, nulls

In [14]:
# Shape -> should be (207480, 8)
print(pairs_staged_df.shape, '\n')

# Check data types
print(pairs_staged_df.dtypes)

(207480, 8) 

occ2010_from      int64
occ2010_to        int64
wage_from       float32
wage_to         float32
wageg_from        int64
wageg_to          int64
ba_share_to     float64
n_trans         float64
dtype: object


In [15]:
# n_trans is NULL wherever a directed pair was never observed in CPS; every other column is complete by construction
null_counts = pairs_staged_df.isna().sum()
null_counts = null_counts[(null_counts > 0)]
print(null_counts) # if any nulls in other cols, something's off

n_trans    189524
dtype: int64


## Transitions

Observed pairs are the positives; unobserved (NULL n_trans) are the negatives for future binary validation testing


In [68]:
# Label each directed pair as observed in CPS or not
observed = pairs_staged_df['n_trans'].notna()

n_obs = observed.sum() # number of observed
n_un_obs = (~observed).sum() # number of unobserved

print(f"observed: {n_obs:,} unobserved: {n_un_obs:,}   unobs per obs: {n_un_obs / n_obs:.1f}")

# Transition volume among observed pairs (weighted counts, in thousands)
observed_trans_df = pairs_staged_df.loc[observed, "n_trans"]
observed_trans_df.describe()

observed: 17,956 unobserved: 189,524   unobs per obs: 10.6


count    17956.000000
mean         0.887212
std          2.751193
min          0.010897
25%          0.168064
50%          0.305713
75%          0.614576
max         88.670913
Name: n_trans, dtype: float64

In [ ]:
# Transitions Distribution Plot
log_volume = np.log10(observed_trans_df) # log transitions so they don't collapse to the first bin

fig = px.histogram(
    x=log_volume,
    nbins=50,
    labels={"x": "log₁₀ annual transitions (thousands)"},
    title="Observed transition volume",
)
fig.update_layout(yaxis_title="Directed pairs", title_x=0.3, bargap=0.02)
fig.show()

## Skills

In [37]:
skills_df

,occ2010,occ2010_title,readingcomprehension,activelistening,writing,speaking,mathematics,science,criticalthinking,activelearning,...,troubleshooting,repairing,qualitycontrolanalysis,judgmentanddecisionmaking,systemsanalysis,systemsevaluation,timemanagement,managementoffinancialresources,managementofmaterialresources,managementofpersonnelresources
0,10,chief executives and legislators,4.1200,4.000000,4.120000,4.250000,3.250000,1.620000,4.380000,3.750000,...,1.500000,1.000000,1.880000,4.750000,4.120000,4.250000,4.000000,4.250000,4.000000,4.250000
1,20,general and operations managers,4.0000,4.000000,3.500000,4.000000,2.620000,1.500000,3.880000,3.620000,...,1.750000,1.000000,2.380000,3.620000,3.120000,3.120000,3.620000,3.000000,3.120000,3.750000
2,40,advertising and promotions managers,3.7500,4.120000,3.750000,4.000000,3.000000,1.620000,4.000000,3.250000,...,1.000000,1.000000,1.620000,3.750000,3.120000,3.120000,3.500000,2.750000,2.620000,3.120000
3,50,marketing and sales managers,3.8800,3.953274,3.475928,3.953274,2.902654,1.670620,3.880000,3.800620,...,1.000000,1.000000,1.800620,3.829380,3.475928,3.573274,3.652654,2.953274,2.699380,3.685308
4,60,public relations and fundraising managers,3.8800,4.000000,3.880000,4.120000,3.000000,1.250000,4.120000,3.380000,...,1.000000,1.000000,1.250000,3.750000,3.120000,3.000000,3.500000,3.250000,2.620000,3.120000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
469,9630,machine feeders and offbearers,2.7500,2.750000,2.380000,2.750000,2.120000,1.250000,2.500000,2.000000,...,2.750000,1.750000,2.750000,2.500000,1.620000,2.000000,2.500000,1.750000,1.880000,1.880000
470,9640,"packers and packagers, hand",2.6200,2.750000,2.000000,2.620000,2.000000,1.000000,2.750000,2.120000,...,1.750000,1.250000,2.250000,2.250000,1.880000,1.880000,2.250000,1.250000,1.500000,1.750000
471,9650,pumping station operators,2.9369,3.001232,2.768825,3.043126,2.296720,1.547028,3.192875,2.632567,...,3.076874,2.969538,2.650057,3.059384,2.226931,2.043126,2.876284,1.637798,1.966129,2.318639
472,9720,refuse and recyclable material collectors,2.6200,2.880000,2.500000,2.880000,1.000000,1.000000,2.750000,2.250000,...,2.500000,2.500000,2.250000,2.380000,1.380000,1.380000,2.500000,1.000000,1.000000,2.000000


In [78]:
skill_cols = [c for c in skills.columns if c not in ("occ2010", "occ2010_title")]
skill_cols

['readingcomprehension',
 'activelistening',
 'writing',
 'speaking',
 'mathematics',
 'science',
 'criticalthinking',
 'activelearning',
 'learningstrategies',
 'monitoring',
 'socialperceptiveness',
 'coordination',
 'persuasion',
 'negotiation',
 'instructing',
 'serviceorientation',
 'complexproblemsolving',
 'operationsanalysis',
 'technologydesign',
 'equipmentselection',
 'installation',
 'programming',
 'operationsmonitoring',
 'operationandcontrol',
 'equipmentmaintenance',
 'troubleshooting',
 'repairing',
 'qualitycontrolanalysis',
 'judgmentanddecisionmaking',
 'systemsanalysis',
 'systemsevaluation',
 'timemanagement',
 'managementoffinancialresources',
 'managementofmaterialresources',
 'managementofpersonnelresources']

In [83]:
# Grab the 35 skill-importance columns
skill_cols = skills_df.drop(columns = ['occ2010', 'occ2010_title']).columns

# skill importance distribution
skill_spread = skills_df[skill_cols].std().sort_values(ascending=False)
print(skill_spread)

operationandcontrol               0.856532
repairing                         0.838581
equipmentmaintenance              0.830365
science                           0.775169
troubleshooting                   0.755572
operationsmonitoring              0.718250
equipmentselection                0.676109
qualitycontrolanalysis            0.639682
operationsanalysis                0.605274
writing                           0.604758
mathematics                       0.567952
systemsanalysis                   0.547832
systemsevaluation                 0.543461
managementofpersonnelresources    0.536480
installation                      0.535722
serviceorientation                0.535320
persuasion                        0.529389
readingcomprehension              0.527540
negotiation                       0.521620
instructing                       0.520432
managementoffinancialresources    0.509506
learningstrategies                0.509242
activelearning                    0.492713
complexprob

In [85]:
# skill importance correlation
corr = skills_df[skill_cols].corr()

fig = px.imshow(
    corr,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="O*NET skill-importance correlation",
)
# Crisp cell edges (no interpolation), every skill labeled, square aspect
fig.update_traces(zsmooth=False)
fig.update_layout(
    width=900, height=900,
    xaxis=dict(tickmode="array", tickvals=list(range(len(skill_cols))), ticktext=skill_cols, tickfont_size=9),
    yaxis=dict(tickmode="array", tickvals=list(range(len(skill_cols))), ticktext=skill_cols, tickfont_size=9),
)
fig.show()